# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DjebrilSVN/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# Setup
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "scikit-learn"], check=True)

import duckdb, os

# Load token securely
token = os.environ.get("HF_TOKEN")
if not token:
    for p in [".env", "../.env", "../../.env"]:
        if os.path.exists(p):
            for line in open(p):
                if line.strip().startswith("HF_TOKEN="):
                    token = line.strip().split("=", 1)[1]
            break
if not token and "google.colab" in sys.modules:
    from google.colab import userdata
    try: token = userdata.get("HF_TOKEN")
    except Exception: pass

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')")

# Mid-panel month
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')"

print("Ready. Token loaded:", bool(token))

Ready. Token loaded: True


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one daily observation for one pseudonymized content item belonging to one pseudonymized client.**
That is the grain: `report_date × client_hash_id × content_hash_id`.

- **Table used:** `fact_content_daily_performance`
- **Time window:** `month=2026-03` (March 1–31 2026) — the safe mid-panel development window
- **What I'd predict (proxy label):** A binary flag for whether a page is in the high-traffic tier (`gsc_impressions > 1000` aggregated over the month) — a cluster proxy: high-traffic pages map to 'champion' archetype in my clustering lane
- **Deliberately excluded:** Any row where `ga4_data_available IS NOT TRUE` — those rows carry zero-filled GA4 columns that masquerade as "no engagement" but are really measurement absences

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Quick sanity check: what columns does the table have?
sample = con.sql(f"SELECT * FROM {REL} LIMIT 1").df()
print(f"Columns ({len(sample.columns)}):")
print(list(sample.columns))

Columns (31):
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Field | Note |
|--------|-------|------|
| **Context** | `report_date`, `client_hash_id`, `content_hash_id` | Identifiers — grouping and joining only, never model inputs |
| **Feature** | `gsc_impressions` | Past visibility — knowable at decision moment (historical total already logged) |
| **Feature** | `gsc_avg_position` | Past average rank — recorded daily, already past at prediction time |
| **Feature** | `ga4_sessions` | Past GA4 sessions — trailing historical count, knowable at decision moment |
| **Feature** | `ga4_engaged_sessions` | Past engaged sessions — trailing historical count, knowable at decision moment |
| **Feature** | `ctr` (derived) | `gsc_clicks / gsc_impressions × 100` — ratio of two past totals, knowable at decision moment |
| **Label / proxy** | `is_high_traffic` | `gsc_impressions > 1000` over the month — the thing we predict; never a feature |
| **Excluded** | `ga4_*` when `ga4_data_available IS NOT TRUE` | Zero-filled missing periods, not real zeros — using them injects noise silently |
| **Excluded (leakage trap)** | `gsc_clicks` (as standalone feature) | Mathematically tied to impressions (our label) — demonstrated in section 3 |

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
pass

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ── Query 1: Grain check — zero rows means the grain holds ────────────────────
print("QUERY 1 — Grain check (expect 0 violations)")
grain = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM {REL}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()
print(f"  Grain violations: {len(grain)} rows  ← should be 0\n")

# ── Query 2: Row count and date span ─────────────────────────────────────────
print("QUERY 2 — Row count and date span")
counts = con.sql(f"""
    SELECT COUNT(*) AS total_rows, MIN(report_date) AS start_date, MAX(report_date) AS end_date
    FROM {REL}
""").df()
print(counts.to_string(index=False), "\n")

# ── Query 3: Availability — filter with IS TRUE, show surviving rows ──────────
print("QUERY 3 — Availability (ga4_data_available IS TRUE)")
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available
    FROM {REL}
""").df()
print(avail.to_string(index=False), "\n")

# ── Five-feature frame + leakage trap ────────────────────────────────────────
print("FIVE-FEATURE FRAME + LEAKAGE TRAP")
frame = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions)      AS gsc_impressions,
        SUM(gsc_clicks)           AS gsc_clicks,
        AVG(gsc_avg_position)     AS gsc_avg_position,
        SUM(ga4_sessions)         AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
    FROM {REL}
    WHERE ga4_data_available IS TRUE AND gsc_data_available IS TRUE
      AND gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id
""").df()

frame["ctr"]             = (frame["gsc_clicks"] / frame["gsc_impressions"]) * 100
frame["is_high_traffic"] = (frame["gsc_impressions"] > 1000).astype(int)
print(f"  Feature frame: {len(frame):,} rows, label balance: {frame['is_high_traffic'].mean():.1%} high-traffic")
print(frame[["gsc_impressions","gsc_avg_position","ga4_sessions","ga4_engaged_sessions","ctr","is_high_traffic"]].head(3).to_string(index=False))

y = frame["is_high_traffic"]
X_trap   = frame[["ga4_sessions","ga4_engaged_sessions","gsc_avg_position","ctr","gsc_clicks"]].fillna(0)
X_honest = frame[["ga4_sessions","ga4_engaged_sessions","gsc_avg_position","ctr"]].fillna(0)

Xt_tr, Xt_te, yt_tr, yt_te = train_test_split(X_trap,   y, test_size=0.2, random_state=42)
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(X_honest, y, test_size=0.2, random_state=42)

trap_score   = accuracy_score(yt_te, LogisticRegression(max_iter=500).fit(Xt_tr, yt_tr).predict(Xt_te))
honest_score = accuracy_score(yh_te, LogisticRegression(max_iter=500).fit(Xh_tr, yh_tr).predict(Xh_te))

print(f"\n  *** Leakage experiment ***")
print(f"  Score WITH gsc_clicks (leaked):    {trap_score:.3f}  <- artificially inflated")
print(f"  Honest score (gsc_clicks removed): {honest_score:.3f}  <- the real number")
print(f"  Leakage premium: {trap_score - honest_score:+.3f}")

QUERY 1 — Grain check (expect 0 violations)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Grain violations: 0 rows  ← should be 0

QUERY 2 — Row count and date span


 total_rows start_date   end_date
    9841378 2026-03-01 2026-03-31 

QUERY 3 — Availability (ga4_data_available IS TRUE)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  ga4_available  gsc_available
    9841378       413966.0      3611061.0 

FIVE-FEATURE FRAME + LEAKAGE TRAP


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Feature frame: 63,856 rows, label balance: 20.8% high-traffic
 gsc_impressions  gsc_avg_position  ga4_sessions  ga4_engaged_sessions      ctr  is_high_traffic
           396.0         21.671343          38.0                   3.0 1.010101                0
             1.0          0.000000           1.0                   0.0 0.000000                0
            62.0          5.317235           7.0                   1.0 8.064516                0



  *** Leakage experiment ***
  Score WITH gsc_clicks (leaked):    0.978  <- artificially inflated
  Honest score (gsc_clicks removed): 0.864  <- the real number
  Leakage premium: +0.114


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation — unbalanced panel history:** The warehouse is not a clean rectangular panel. Each client's history starts on a different date (`dim_clients.gsc_data_start`). Clients who joined the platform late contribute only a few months, while older clients contribute up to 17 months. Aggregating across the full panel without per-client normalisation conflates *"low impressions because new client"* with *"low impressions because declining page"* — exactly the kind of silent mistake that produces wrong clusters in Lane 3.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Demonstrate the limitation: days observed per client in March 2026 vary wildly
client_depth = con.sql(f"""
    SELECT client_hash_id, COUNT(DISTINCT report_date) AS days_observed
    FROM {REL}
    GROUP BY client_hash_id
    ORDER BY days_observed
""").df()

print(f"Clients in March 2026:              {len(client_depth)}")
print(f"Min days observed:                  {client_depth['days_observed'].min()}")
print(f"Max days observed:                  {client_depth['days_observed'].max()}")
print(f"Clients with fewer than 31 days:    {(client_depth['days_observed'] < 31).sum()}")

Clients in March 2026:              55
Min days observed:                  9
Max days observed:                  31
Clients with fewer than 31 days:    4


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.